# 07 — Random Forest Model

This notebook fits a conservative Random Forest using the frozen Phase 3 predictors and split.
Preprocessing is fit on training data only. Validation selects among four small configurations;
the fixed configuration is then refit through 2021 and evaluated once on 2022.


## 1. Inputs and dependency verification


In [ ]:
from pathlib import Path
import importlib.util
import json
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "phase4_utils.py").exists():
            return candidate
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = locate_root(Path.cwd())
spec = importlib.util.spec_from_file_location("phase4_utils", PROJECT_ROOT / "src" / "phase4_utils.py")
u = importlib.util.module_from_spec(spec)
spec.loader.exec_module(u)
evaluation = u.load_evaluation_module(PROJECT_ROOT)
splits = u.load_splits(PROJECT_ROOT)
print({name: data["combined"].shape for name, data in splits.items()})


In [ ]:
for split_name, data in splits.items():
    assert data["X"][u.KEYS].equals(data["y"][u.KEYS])
    assert sorted(data["X"]["year"].unique().tolist()) == u.SPLIT_YEARS[split_name]
    assert data["X"]["region"].nunique() == 13
    assert not data["X"].duplicated(u.KEYS).any()
    assert list(data["X"].columns) == u.KEYS + u.PREDICTORS
print("Frozen protocol and target alignment verified.")


In [ ]:
try:
    import sklearn
    import joblib
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestRegressor
except ImportError as exc:
    raise ImportError("Notebook 07 requires scikit-learn and joblib.") from exc

MODEL_DIR = PROJECT_ROOT / "models" / "random_forest"
RESULT_DIR = PROJECT_ROOT / "results" / "random_forest"
FIGURE_DIR = PROJECT_ROOT / "figures" / "random_forest"
for directory in (MODEL_DIR, RESULT_DIR, FIGURE_DIR): directory.mkdir(parents=True, exist_ok=True)
u.write_json(RESULT_DIR / "environment.json", u.package_versions(
    ["numpy", "pandas", "sklearn", "joblib", "matplotlib", "seaborn"]))

MODEL_COLUMNS = ["region"] + u.PREDICTORS
NUMERIC = u.PREDICTORS
def make_pipeline(params):
    preprocess = ColumnTransformer([
        ("region", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["region"]),
        ("numeric", "passthrough", NUMERIC),
    ], remainder="drop")
    model = RandomForestRegressor(random_state=u.RANDOM_SEED, n_jobs=-1, **params)
    return Pipeline([("preprocess", preprocess), ("model", model)])

CANDIDATES = [
    {"n_estimators": 150, "max_depth": 2, "min_samples_leaf": 2, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": 3, "min_samples_leaf": 2, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": None, "min_samples_leaf": 2, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": 3, "min_samples_leaf": 1, "max_features": 1.0},
]


## 2. Validation-only configuration selection


In [ ]:
train, val, test = splits["train"]["combined"], splits["validation"]["combined"], splits["test"]["combined"]
candidate_rows = []
for index, params in enumerate(CANDIDATES):
    pipeline = make_pipeline(params)
    pipeline.fit(train[MODEL_COLUMNS], train[u.TARGET])
    predicted = pipeline.predict(val[MODEL_COLUMNS])
    metrics = evaluation.evaluate_regression(val[u.TARGET], predicted,
                                             model_name=f"RF_candidate_{index+1}", split="validation")
    metrics["candidate_id"] = index + 1
    candidate_rows.append(metrics)
candidate_metrics = pd.DataFrame(candidate_rows).sort_values(["RMSE", "candidate_id"]).reset_index(drop=True)
selected_id = int(candidate_metrics.iloc[0]["candidate_id"])
selected_params = CANDIDATES[selected_id - 1]
display(candidate_metrics); print("Selected:", selected_params)


## 3. Validation and final test fit


In [ ]:
validation_model = make_pipeline(selected_params)
validation_model.fit(train[MODEL_COLUMNS], train[u.TARGET])
validation_pred = validation_model.predict(val[MODEL_COLUMNS])
validation_output = u.prediction_frame(val[u.KEYS], val[u.TARGET], validation_pred,
                                       "Random Forest", "validation")

development = pd.concat([train, val], ignore_index=True).sort_values(u.KEYS)
final_model = make_pipeline(selected_params)
final_model.fit(development[MODEL_COLUMNS], development[u.TARGET])
test_pred = final_model.predict(test[MODEL_COLUMNS])
test_output = u.prediction_frame(test[u.KEYS], test[u.TARGET], test_pred, "Random Forest", "test")

predictions = pd.concat([validation_output, test_output], ignore_index=True)
metrics = pd.DataFrame([u.evaluate_prediction_frame(evaluation, validation_output),
                        u.evaluate_prediction_frame(evaluation, test_output)])
predictions.to_csv(RESULT_DIR / "predictions.csv", index=False, float_format="%.15g")
metrics.to_csv(RESULT_DIR / "metrics.csv", index=False, float_format="%.15g")
candidate_metrics.to_csv(RESULT_DIR / "candidate_validation_metrics.csv", index=False)
joblib.dump(validation_model, MODEL_DIR / "validation_model.joblib")
joblib.dump(final_model, MODEL_DIR / "random_forest_model.joblib")
u.write_json(MODEL_DIR / "parameters.json", {"random_seed": u.RANDOM_SEED,
    "selected_candidate_id": selected_id, "parameters": selected_params,
    "selection_split": "validation", "test_refit_years": [2019, 2020, 2021]})

preprocessor = final_model.named_steps["preprocess"]
names = preprocessor.get_feature_names_out()
importance = pd.DataFrame({"feature": names,
    "importance": final_model.named_steps["model"].feature_importances_}).sort_values("importance", ascending=False)
importance.to_csv(RESULT_DIR / "feature_importance.csv", index=False)
display(metrics); display(importance.head(15))


## 4. Figures and limitations


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5.5))
for ax, split_name in zip(axes,["validation","test"]):
    group=predictions.query("dataset_split == @split_name").sort_values("actual")
    ax.plot(group["actual"].to_numpy()/1e9,marker="o",label="Observed")
    ax.plot(group["predicted"].to_numpy()/1e9,marker="s",label="Random Forest")
    ax.set_title(split_name.title()); ax.set_ylabel("Billion kWh")
axes[1].legend(frameon=False); fig.tight_layout()
fig.savefig(FIGURE_DIR/"predictions.png",bbox_inches="tight"); plt.show()

fig,ax=plt.subplots(figsize=(9,5)); sns.scatterplot(data=predictions,x="predicted",y="residual",
    hue="dataset_split",ax=ax); ax.axhline(0,color="black",linewidth=1)
ax.set_title("Random Forest Residuals",weight="bold"); fig.tight_layout()
fig.savefig(FIGURE_DIR/"residuals.png",bbox_inches="tight"); plt.show()

top=importance.head(15).sort_values("importance")
fig,ax=plt.subplots(figsize=(10,7)); sns.barplot(data=top,x="importance",y="feature",color="#2A9D8F",ax=ax)
ax.set_title("Random Forest Feature Importance",weight="bold"); fig.tight_layout()
fig.savefig(FIGURE_DIR/"feature_importance.png",bbox_inches="tight"); plt.show()

report=f'''# Random Forest Summary

- Random seed: {u.RANDOM_SEED}
- Candidate configurations: {len(CANDIDATES)}
- Selected using: 2021 validation RMSE
- Final refit: 2019–2021
- Test: 2022 once

The dataset is extremely small for a flexible ensemble, so depth and leaf size are constrained.
Feature importance describes predictive split usage, not causal influence. Regional variability,
short history, and correlated lag features can make importance unstable.
'''
(RESULT_DIR/"summary_report.md").write_text(report,encoding="utf-8")
print("Notebook 07 complete.")
